# Feature Selection and Machine Learning
Try to classify rejection vs no-rejection using radiomics features.

Given the statistical analysis results (no features significant on full dataset,
2 features marginally significant in the late subset), we don't expect strong ML
performance. This notebook documents the attempt.

Steps:
1. Remove highly correlated features (|r| > 0.9)
2. Train classifiers with stratified cross-validation
3. Report AUC, sensitivity, specificity
4. Try both full dataset and late-only subset

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, roc_auc_score, recall_score
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")

In [ ]:
# load merged data
df = pd.read_csv(os.path.join("reports", "merged_radiomics_clinical.csv"))
feature_cols = [c for c in df.columns if c.startswith("original_")]

print(f"Samples: {len(df)}, Features: {len(feature_cols)}")

## Step 1: Remove highly correlated features
Drop one feature from each pair with |r| > 0.9 to reduce redundancy.

In [ ]:
def remove_correlated_features(data, features, threshold=0.9):
    """Drop features that are highly correlated with another feature."""
    corr = data[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        if any(upper[col] > threshold):
            to_drop.add(col)

    remaining = [f for f in features if f not in to_drop]
    return remaining, to_drop

reduced_features, dropped = remove_correlated_features(df, feature_cols)
print(f"Original features: {len(feature_cols)}")
print(f"Dropped (correlated): {len(dropped)}")
print(f"Remaining features: {len(reduced_features)}")

## Step 2: Classification on full dataset

In [ ]:
def evaluate_classifiers(X, y, label):
    """Run stratified 5-fold CV with Logistic Regression and Random Forest."""
    print(f"\n--- {label} ---")
    print(f"Samples: {len(y)}, Features: {X.shape[1]}")
    print(f"Class balance: {(y == 0).sum()} no-rej, {(y == 1).sum()} rej")

    if (y == 1).sum() < 5 or (y == 0).sum() < 5:
        print("Too few samples in one class. Skipping.")
        return None

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "auc": "roc_auc",
        "sensitivity": make_scorer(recall_score, pos_label=1),
        "specificity": make_scorer(recall_score, pos_label=0),
    }

    models = {
        "LogReg_L2": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, penalty="l2", class_weight="balanced",
                                       max_iter=1000, random_state=42)),
        ]),
        "RandomForest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                           random_state=42)),
        ]),
    }

    all_results = []
    for name, model in models.items():
        cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)
        auc_mean = cv_results["test_auc"].mean()
        auc_std = cv_results["test_auc"].std()
        sens_mean = cv_results["test_sensitivity"].mean()
        spec_mean = cv_results["test_specificity"].mean()

        print(f"  {name}: AUC={auc_mean:.3f} (+/-{auc_std:.3f}), "
              f"Sens={sens_mean:.3f}, Spec={spec_mean:.3f}")

        all_results.append({
            "dataset": label,
            "model": name,
            "auc_mean": round(auc_mean, 3),
            "auc_std": round(auc_std, 3),
            "sensitivity": round(sens_mean, 3),
            "specificity": round(spec_mean, 3),
        })

    return all_results

In [ ]:
# full dataset
X_full = df[reduced_features].values
y_full = df["rejection"].values

results_full = evaluate_classifiers(X_full, y_full, "Full dataset (N=133)")

## Step 3: Classification on late subset (motivo 3-5)
This is where the statistical analysis found marginal signal.

In [ ]:
# late subset (beyond 90 days)
late = df[df["motivo"].isin([3, 4, 5])]
X_late = late[reduced_features].values
y_late = late["rejection"].values

results_late = evaluate_classifiers(X_late, y_late, "Late subset motivo 3-5 (N={0})".format(len(late)))

## Step 4: Feature importance from Random Forest
Train on full dataset and extract feature importances.

In [ ]:
# train random forest on full data to get feature importances
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[reduced_features].values)
rf.fit(X_scaled, df["rejection"].values)

importances = pd.DataFrame({
    "feature": reduced_features,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 15 features by Random Forest importance:")
print(importances.head(15).to_string(index=False))

In [ ]:
# save all results
all_results = []
if results_full:
    all_results.extend(results_full)
if results_late:
    all_results.extend(results_late)

if all_results:
    results_table = pd.DataFrame(all_results)
    output_path = os.path.join("reports", "ml_results.csv")
    results_table.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
    print()
    print(results_table.to_string(index=False))

# save feature importances
importances.to_csv(os.path.join("reports", "feature_importances_rf.csv"), index=False)
print("\nSaved feature importances to reports/feature_importances_rf.csv")